# 🛡️ Retention Strategy & CLV Analysis

Score all customers with churn probability.
Segment into risk tiers. Compute Customer Lifetime Value.
Generate actionable retention recommendations.

In [ ]:
import sys
sys.path.insert(0, "../src")
import joblib, pandas as pd, numpy as np, matplotlib.pyplot as plt
from data_loader import load_config, load_raw_data, clean_data
from features import engineer_features, encode_features, prepare_features
from retention import compute_clv, segment_customers, generate_retention_report
config = load_config("../config.yaml")
config["data"]["raw_path"] = "../" + config["data"]["raw_path"]
config["model"]["model_path"] = "../" + config["model"]["model_path"]
df = load_raw_data(config["data"]["raw_path"])
df_clean = clean_data(df, config)
model = joblib.load(config["model"]["model_path"])
print("Model loaded")

In [ ]:
_, _, _, _, feature_names = prepare_features(df_clean, config)
df_fe = engineer_features(df_clean)
df_enc = encode_features(df_fe, config)
drop_cols = config["features"]["drop_cols"] + [config["data"]["target_column"]]
X_full = df_enc.drop(columns=drop_cols, errors="ignore")
X_full = X_full.reindex(columns=feature_names, fill_value=0)
probs = model.predict_proba(X_full)[:, 1]
df_result = df_clean.copy()
df_result["churn_prob"] = probs
df_result["clv"] = compute_clv(df_result, config)
df_result = segment_customers(df_result, config)
print(df_result["risk_tier"].value_counts())

In [ ]:
generate_retention_report(df_result, save_dir="../reports")
from IPython.display import Image
Image("../reports/retention_overview.png")

In [ ]:
# Top 10 High Risk customers by CLV
high_risk = df_result[df_result["risk_tier"]=="🔴 High Risk"].nlargest(10, "clv")
print("Top 10 High-Risk Customers by CLV:")
high_risk[["tenure","MonthlyCharges","Contract","churn_prob","clv","risk_tier"]].to_string()